<a href="https://colab.research.google.com/github/sreejit-19/IQF/blob/main/AMARTYA_FINAL_CODE_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.metrics import r2_score
from scipy.stats import spearmanr

############################################################
# 1. LOAD DATA
############################################################

df = pd.read_csv('Sorted_Asset_Pricing_Data.csv')

# Crucial: Sort chronologically per asset for flawless rolling windows
df = df.sort_values(by=['Co_Name', 'Month']).reset_index(drop=True)

############################################################
# 2. FEATURE ENGINEERING (CORRECTED & LEAK-PROOF)
############################################################

# Basic factors
base_features = ['Size', 'BM', 'Momentum']

# Feature Interactions
df['BM_x_Momentum'] = df['BM'] * df['Momentum']
df['Size_x_Momentum'] = df['Size'] * df['Momentum']

# FIX 1: Volatility must ONLY use past returns to stop look-ahead leak
df['Historical_Return'] = df.groupby('Co_Name')['Stock_Return'].shift(1)
df['Volatility_3M'] = (
    df.groupby('Co_Name')['Historical_Return']
    .transform(lambda x: x.rolling(window=3).std())
)

# FIX 2: Create the True Predictive Target (Predicting Month T+1 using Month T)
df['Next_Month_Return'] = df.groupby('Co_Name')['Stock_Return'].shift(-1)

# FIX 3: Drop the very last month of the dataset (Dec 2023)
# Because it has no future month to predict or validate against
df = df.dropna(subset=['Next_Month_Return']).reset_index(drop=True)

# Impute remaining missing values safely (like the first 3 months of volatility)
for col in df.columns:
    if df[col].dtype != 'object' and col != 'Next_Month_Return':
        df[col] = df[col].fillna(df[col].median())

features = [
    'Size',
    'BM',
    'Momentum',
    'BM_x_Momentum',
    'Size_x_Momentum',
    'Volatility_3M'
]

############################################################
# 3. MONTHLY CROSS-SECTIONAL RANK NORMALIZATION
############################################################

def rank_normalize(group):
    group = group.copy()
    for col in features:
        group[col] = group[col].rank(pct=True) - 0.5
    return group

df_norm = df.groupby('Month', group_keys=False).apply(rank_normalize)

############################################################
# 4. CREATE CLASSIFICATION TARGET
############################################################

def create_target(group):
    group = group.copy()
    threshold = group['Next_Month_Return'].quantile(0.80)
    group['Target'] = (group['Next_Month_Return'] >= threshold).astype(int)
    return group

df_norm = df_norm.groupby('Month', group_keys=False).apply(create_target)

############################################################
# 5. WALK-FORWARD VALIDATION SETTINGS
############################################################

results = []
all_predictions = []
train_window = 3
years = sorted(df_norm['Year'].unique())

############################################################
# 6. WALK-FORWARD LOOP
############################################################

for test_year in years:
    train_start = test_year - train_window
    train_end = test_year - 1

    if train_start < years[0]:
        continue

    train = df_norm[(df_norm['Year'] >= train_start) & (df_norm['Year'] <= train_end)]
    test = df_norm[df_norm['Year'] == test_year].copy()

    if len(test) == 0:
        continue

    X_train, y_train = train[features], train['Target']
    X_test, y_test = test[features], test['Target']

    ########################################################
    # 7. XGBOOST MODEL (CONSTRAINED AGAINST OVERFITTING)
    ########################################################
    model = xgb.XGBClassifier(
        objective='binary:logistic',
        n_estimators=450,
        learning_rate=0.01,
        max_depth=3,
        reg_lambda=4.0,
        reg_alpha=1.5,
        subsample=0.9,
        colsample_bytree=0.9,
        random_state=42,
        eval_metric='logloss'
    )
    model.fit(X_train, y_train)

    ########################################################
    # 8. EXTRACT PROBABILITIES
    ########################################################
    test['Prediction'] = model.predict_proba(X_test)[:, 1]
    all_predictions.append(test)

    ########################################################
    # 9. MONTHLY LONG-SHORT PORTFOLIO TRADING
    ########################################################
    for month, group in test.groupby('Month'):
        ranked = group.sort_values(by='Prediction', ascending=False)

        # Build Top 20% and Bottom 20% portfolios
        n = max(5, int(len(ranked) * 0.2))
        long_port = ranked.head(n)
        short_port = ranked.tail(n)

        # FIX 4: Evaluate performance on the actual future trade period
        long_ret = long_port['Next_Month_Return'].mean()
        short_ret = short_port['Next_Month_Return'].mean()
        ls_ret = long_ret - short_ret

        # Rank correlation between predictions and realized forward returns
        ic = spearmanr(group['Prediction'], group['Next_Month_Return'])[0]

        results.append({
            'Year': test_year,
            'Month': month,
            'LS_Return': ls_ret,
            'IC': ic
        })

############################################################
# 10. COMBINE RESULTS
############################################################
results_df = pd.DataFrame(results)
all_predictions_df = pd.concat(all_predictions)

############################################################
# 11. METRIC COMPUTATIONS
############################################################
r2_like = r2_score(all_predictions_df['Target'], all_predictions_df['Prediction'])
results_df['Cumulative_Return'] = (1 + results_df['LS_Return']).cumprod() - 1
final_return = results_df['Cumulative_Return'].iloc[-1]

mean_ret = results_df['LS_Return'].mean()
std_ret = results_df['LS_Return'].std()
sharpe = (mean_ret / std_ret) * np.sqrt(12) if std_ret != 0 else 0
avg_ic = results_df['IC'].mean()

############################################################
# 12. PRINT HONEST RESULTS
############################################################
print('=' * 60)
print('XGBOOST ASSET PRICING RESULTS (UN-CHEATED)')
print('=' * 60)
print(f'Pseudo R2: {r2_like:.4f}')
print(f'Total Long-Short Return: {final_return:.2%}')
print(f'Annualized Sharpe Ratio: {sharpe:.4f}')
print(f'Average Information Coefficient (IC): {avg_ic:.4f}')

############################################################
# 13. FEATURE IMPORTANCE
############################################################
print('\nFeature Importance Breakdown:')
importances = sorted(
    dict(zip(features, model.feature_importances_)).items(),
    key=lambda x: x[1],
    reverse=True
)
for feature, importance in importances:
    print(f' - {feature}: {importance:.4f}')

/tmp/ipykernel_18127/126657331.py:65: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_norm = df.groupby('Month', group_keys=False).apply(rank_normalize)
/tmp/ipykernel_18127/126657331.py:77: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_norm = df_norm.groupby('Month', group_keys=False).apply(create_target)


XGBOOST ASSET PRICING RESULTS (UN-CHEATED)
Pseudo R2: -0.0097
Total Long-Short Return: 8.58%
Annualized Sharpe Ratio: 0.2514
Average Information Coefficient (IC): -0.0001

Feature Importance Breakdown:
 - Momentum: 0.1923
 - Size: 0.1793
 - BM_x_Momentum: 0.1672
 - Size_x_Momentum: 0.1590
 - BM: 0.1559
 - Volatility_3M: 0.1464


In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import spearmanr
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

############################################################
# 1. LOAD & SORT
############################################################
df = pd.read_csv('Sorted_Asset_Pricing_Data.csv')
df = df.sort_values(['Co_Name', 'Month']).reset_index(drop=True)

############################################################
# 2. FEATURE ENGINEERING (leak-proof)
# All stock-level features use shift(1) — only past
# information is used to predict next month's return.
############################################################
df['Ret_1M'] = df.groupby('Co_Name')['Stock_Return'].shift(1)
df['Ret_3M'] = df.groupby('Co_Name')['Stock_Return'] \
                 .transform(lambda x: x.shift(1).rolling(3).sum())
df['Vol_3M'] = df.groupby('Co_Name')['Stock_Return'] \
                 .transform(lambda x: x.shift(1).rolling(3).std())
df['MomVol']   = df['Momentum'] / (df['Vol_3M'] + 1e-8)
df['BM_x_Mom'] = df['BM'] * df['Momentum']
df['Sz_x_Mom'] = df['Size'] * df['Momentum']

# Target: next month's return
df['Next_Return'] = df.groupby('Co_Name')['Stock_Return'].shift(-1)
df = df.dropna(subset=['Next_Return']).reset_index(drop=True)

# Impute early NaNs from rolling windows
num_cols = df.select_dtypes(include=np.number).columns.difference(['Next_Return'])
for c in num_cols:
    df[c] = df[c].fillna(df[c].median())

############################################################
# 3. FEATURES
############################################################
features = [
    'Size', 'BM', 'Momentum',
    'Ret_1M', 'Ret_3M', 'Vol_3M',
    'MomVol', 'BM_x_Mom', 'Sz_x_Mom'
]

############################################################
# 4. CROSS-SECTIONAL RANK NORMALIZATION
# Rank each stock within its month — removes outliers and
# makes features comparable across different time periods.
############################################################
def rank_normalize(group, cols):
    g = group.copy()
    for c in cols:
        g[c] = g[c].rank(pct=True) - 0.5
    return g

df_norm = df.groupby('Month', group_keys=False) \
            .apply(lambda g: rank_normalize(g, features))
df_norm['Month'] = df['Month'].values
df_norm = df_norm.reset_index(drop=True)

############################################################
# 5. SIMPLE IN/OUT OF SAMPLE SPLIT
# Train: 2018-2022 (5 years, 3000 observations)
# Test:  2023      (1 year,  600 observations)
# No future data ever touches training — clean temporal split.
############################################################
train = df_norm[df_norm['Year'] <= 2022]
test  = df_norm[df_norm['Year'] == 2023].copy()

print(f"Train: 2018-2022 | {len(train)} rows | {train['Month'].nunique()} months")
print(f"Test:  2023      | {len(test)} rows  | {test['Month'].nunique()} months")

############################################################
# 6. XGBOOST REGRESSOR
# Predicts next month's return directly (regression).
# Heavy regularization for small 50-stock universe:
#   min_child_weight=10  no leaf from fewer than 10 samples
#   reg_lambda=5         strong L2 penalty on leaf weights
#   reg_alpha=2          L1 sparsity
#   max_depth=3          max 3-way interactions only
############################################################
model = xgb.XGBRegressor(
    objective='reg:squarederror',
    n_estimators=300,
    learning_rate=0.01,
    max_depth=3,
    reg_lambda=5.0,
    reg_alpha=2.0,
    subsample=0.8,
    colsample_bytree=0.7,
    min_child_weight=10,
    random_state=42,
    verbosity=0
)
model.fit(train[features], train['Next_Return'])

############################################################
# 7. PREDICT & BUILD MONTHLY LONG-SHORT PORTFOLIO
# Each month in 2023:
#   - Rank all 50 stocks by predicted return
#   - Long top 20% (10 stocks), short bottom 20% (10 stocks)
#   - Record actual realized return of this portfolio
############################################################
test = test.copy()
test['Pred'] = model.predict(test[features])

results = []
for month, g in test.groupby('Month'):
    ranked = g.sort_values('Pred', ascending=False)
    n      = max(5, int(len(ranked) * 0.2))
    ls_ret = (ranked.head(n)['Next_Return'].mean() -
              ranked.tail(n)['Next_Return'].mean())
    ic     = spearmanr(g['Pred'], g['Next_Return'])[0]
    results.append({'Month': month, 'LS': ls_ret, 'IC': ic})

############################################################
# 8. PERFORMANCE METRICS
############################################################
results_df   = pd.DataFrame(results)
results_df['CumRet'] = (1 + results_df['LS']).cumprod() - 1

# In-sample IC (how well model fits training data)
in_sample_pred = model.predict(train[features])
in_ic = spearmanr(in_sample_pred, train['Next_Return'])[0]

# Out-of-sample metrics (what actually matters)
avg_ic   = results_df['IC'].mean()
ic_std   = results_df['IC'].std()
icir     = avg_ic / ic_std if ic_std else 0
mean_ls  = results_df['LS'].mean()
std_ls   = results_df['LS'].std()
sharpe   = mean_ls / std_ls * np.sqrt(12) if std_ls else 0
win_rate = (results_df['LS'] > 0).mean()
final_ret = results_df['CumRet'].iloc[-1]

############################################################
# 9. PRINT RESULTS
############################################################
print('\n' + '=' * 55)
print('XGBOOST ASSET PRICING — HONEST RESULTS')
print('Train: 2018-2022  |  Test: 2023')
print('=' * 55)

print('\n--- Overfitting Check ---')
print(f'In-sample IC  (2018-2022): {in_ic:.4f}')
print(f'Out-of-sample IC (2023):   {avg_ic:.4f}')
print(f'IC Ratio (out/in):         {avg_ic/in_ic:.3f}')
print('(Ratio close to 1 = good generalisation)')
print('(Ratio close to 0 = overfitting)')

print('\n--- Out-of-Sample Performance (2023) ---')
print(f'Total LS Return:      {final_ret:.2%}')
print(f'Annualized Sharpe:    {sharpe:.3f}')
print(f'Average IC:           {avg_ic:.4f}')
print(f'ICIR:                 {icir:.3f}')
print(f'Monthly Win Rate:     {win_rate:.1%}')

print('\n--- Monthly Breakdown (2023) ---')
print(results_df[['Month', 'LS', 'IC']].to_string(index=False))

print('\n--- Feature Importances (by Gain) ---')
imp = pd.Series(
    model.get_booster().get_score(importance_type='gain')
).sort_values(ascending=False)
print(imp.round(4).to_string())

Train: 2018-2022 | 3000 rows | 60 months
Test:  2023      | 550 rows  | 11 months

XGBOOST ASSET PRICING — HONEST RESULTS
Train: 2018-2022  |  Test: 2023

--- Overfitting Check ---
In-sample IC  (2018-2022): 0.2273
Out-of-sample IC (2023):   -0.0639
IC Ratio (out/in):         -0.281
(Ratio close to 1 = good generalisation)
(Ratio close to 0 = overfitting)

--- Out-of-Sample Performance (2023) ---
Total LS Return:      -14.87%
Annualized Sharpe:    -2.156
Average IC:           -0.0639
ICIR:                 -0.469
Monthly Win Rate:     27.3%

--- Monthly Breakdown (2023) ---
     Month        LS        IC
2023-01-01 -0.037246 -0.110204
2023-02-01  0.009382 -0.001008
2023-03-01 -0.022812 -0.010708
2023-04-01 -0.039379 -0.394094
2023-05-01 -0.023546 -0.121537
2023-06-01 -0.041532 -0.058535
2023-07-01  0.018569  0.057191
2023-08-01 -0.028247 -0.041441
2023-09-01 -0.009664 -0.151116
2023-10-01 -0.003751  0.129412
2023-11-01  0.021049 -0.000720

--- Feature Importances (by Gain) ---
Size     